# step6 — 처방 (Prescription): 값 조향 vs Spotlight

**무엇을 확인?** 절벽 바닥(camel 지침 + 앞코드 전부 snake 위반)에서, **그 층 Value에 camel 방향을 밀면** 준수가 되살아나고(우리), **어텐션을 키우면(Spotlight)** 안 된다.

**측정:** camel 선호점수 회복(교사강제, 4모델 공통) · Spotlight는 **지시어 어텐션 비중이 실제 올랐는지**도 함께(구현 탓 반박 차단).

> ⚠️ 엔진의 step6 개입(`InterventionKind.VALUE_ADD`, Spotlight 어텐션 훅)이 붙어야 돈다. deepseek-6.7b는 T4에서 8bit 필요할 수 있음.

In [ ]:
# ① 환경 설정 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes
import transformers; transformers.logging.set_verbosity_error()
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)

In [ ]:
# ② 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step6/steer
!git checkout step6/steer
!git pull --quiet origin step6/steer
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# ③ 조건 설정 — 모델 하나 고르고, 4방법 x 세기 스윈 x 묶음
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation, Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]
# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]

L_STAR = {'qwen':25, 'deepseek':20, 'llama':15, 'stability':18}[MODEL.family]  # step3 피크(맞는 층)
L_WRONG = 5   # 엉뚱한 층(초반)

STRENGTHS = [0.5, 1.0, 2.0, 4.0, 8.0]   # 값 조향 세기
PSI       = [0.1, 0.2, 0.4, 0.6]        # Spotlight 목표 비중(논문 기본 0.1 + 스윈)
BLOCKS    = list(range(42))

def base_pre(block):
    return PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL, pool_block=block)  # 절벽 바닥
def base_ins():
    return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def cond(method, block, strength=None, psi=None):
    if method == 'none':
        iv = Intervention(kind=InterventionKind.NONE)
    elif method == 'value_star':
        iv = Intervention(kind=InterventionKind.VALUE_ADD, layers=[L_STAR], strength=strength, steer_source='code_contrast', target='code')
    elif method == 'value_wrong':
        iv = Intervention(kind=InterventionKind.VALUE_ADD, layers=[L_WRONG], strength=strength, steer_source='code_contrast', target='code')
    elif method == 'spotlight':
        iv = Intervention(kind=InterventionKind.ATTENTION_AMPLIFY, layers='all', amplify=psi, target='instruction')
    tag = method + (f'-s{strength}' if strength is not None else '') + (f'-p{psi}' if psi is not None else '')
    return Condition(model=MODEL, preceding=base_pre(block), instruction=base_ins(),
                     intervention=iv, token_unit='mean', seed=42, tag=tag)

conditions  = [cond('none', b) for b in BLOCKS]
conditions += [cond('value_star',  b, strength=s) for s in STRENGTHS for b in BLOCKS]
conditions += [cond('value_wrong', b, strength=s) for s in STRENGTHS for b in BLOCKS]
conditions += [cond('spotlight',   b, psi=p)      for p in PSI       for b in BLOCKS]
print('모델:', MODEL.family, '| L*=', L_STAR, '| 조건 수:', len(conditions))

In [ ]:
# ④ 실행 — 개입 후 camel 선호점수 회복 측정. 즉시 저장(재개).
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

STEP = 'step6_steer'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers} | GQA {handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RX'))
        if i % 20 == 0 or i == len(todo):
            rec = out.metrics.extra.get('recovery')
            print(f'    [{i}/{len(todo)}] {c.tag}: 회복 {rec if rec is None else round(rec,2)}')
    print('  완료.')
else:
    print('  이미 다 됨')

In [ ]:
# ⑤ 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step6_steer')) for c in conditions
        if result_path(c, step='step6_steer').exists()]
print('불러온 조건:', len(recs), '-> results/step6_steer/')

In [ ]:
# ⑥ 요약 — 비교표(방법별 최고 회복) + 세기별 회복 그림 + 어텐션 비중 검증(①)
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

agg = defaultdict(lambda: defaultdict(list))
attn = defaultdict(lambda: defaultdict(list))
for r in recs:
    ex = r.metrics.extra; tag = r.condition.tag or 'none'
    method = tag.split('-')[0]
    s = tag.split('-',1)[1] if '-' in tag else 'base'
    if ex.get('recovery') is not None: agg[method][s].append(ex['recovery'])
    if ex.get('attn_span_after') is not None: attn[method][s].append(ex['attn_span_after'])

def mean_ci(v):
    a=np.array(v); p=a.mean(); n=len(a); return p, (1.96*a.std(ddof=1)/np.sqrt(n) if n>1 else 0)

rows=[]
for method in ['none','spotlight','value_wrong','value_star']:
    best=None
    for s,vals in agg.get(method,{}).items():
        m,ci=mean_ci(vals)
        if best is None or m>best[1]: best=(s,m,ci,len(vals))
    if best: rows.append({'방법':method,'최고세기':best[0],'회복':round(best[1],3),'CI':round(best[2],3),'n':best[3]})
print('=== 비교표 (방법별 최고 회복) ===')
print(pd.DataFrame(rows).to_string(index=False))

fig,ax=plt.subplots(figsize=(6,4))
for method,color,lab in [('value_star','#1f77b4','value @ L* (ours)'),('value_wrong','#888888','value @ wrong layer')]:
    xs=sorted(agg[method], key=lambda z: float(z.replace('s','')))
    if not xs: continue
    ys,es=zip(*[mean_ci(agg[method][x]) for x in xs])
    ax.errorbar([float(x.replace('s','')) for x in xs], ys, yerr=es, marker='o', color=color, label=lab, capsize=2)
sp=[mean_ci(v)[0] for v in agg.get('spotlight',{}).values()]
if sp: ax.axhline(max(sp), ls='--', color='#d62728', label='Spotlight (best)')
ax.set_xlabel('steering strength'); ax.set_ylabel('compliance recovery (95% CI)')
ax.set_title(f'step6 {MODEL.family}: value-steering vs Spotlight'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

if attn.get('spotlight'):
    print('\n=== ① Spotlight 검증: 개입 후 지시어 어텐션 비중(목표까지 올랐나) ===')
    for p,vals in sorted(attn['spotlight'].items()):
        print(f'  {p}: 실제 비중 {np.mean(vals):.3f} (올랐으면 장치는 작동 → 그래도 회복 안되면 어텐션이 손잡이 아님)')

In [ ]:
# ⑦ 결과 폴더 zip 다운로드
import shutil
shutil.make_archive('step6_steer_results', 'zip', 'results/step6_steer')
try:
    from google.colab import files; files.download('step6_steer_results.zip')
except Exception:
    print('results/step6_steer -> step6_steer_results.zip')